# 빈틈사이 감정분류 — KcELECTRA 임베딩 → 6감정 → 4공감모드

**파이프라인**: KcELECTRA(freeze) 임베딩 → 분류기(LogReg / LinearSVM / XGBoost) 비교 → 6감정 분류 → 4공감모드 매핑

**평가지표 6종** (기획안 기준): ① 정확도(Accuracy) ② 혼동행렬(Confusion Matrix) ③ 정밀도(Precision) ④ 재현율(Recall) ⑤ F1-score ⑥ ROC-AUC
→ 클래스별 P/R/F1 리포트와 함께, **6감정**과 **4공감모드(실KPI)** 둘 다 평가한다.

- 6감정: 분노·슬픔·불안·상처·당황·기쁨  
- 4모드: 화남·속상·계획·응원  
- 매핑: 분노→화남, 슬픔→속상, 불안→계획, 상처→응원, 당황→속상, 기쁨→응원

> 참고: 파인튜닝은 천장이 약 **0.74**였다. 본 노트북은 가벼운 freeze-임베딩 대안의 '최적 버전'이며, 목표선은 그 현실을 반영해 잡는다.

## 0. 설정

In [ ]:
# 필요시: pip install torch transformers scikit-learn xgboost joblib pandas matplotlib
import os, json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "../../data/kcelectra_train_clean.jsonl"  # {"text","emotion"} jsonl
OUT_DIR   = "artifacts"
MODEL_NAME= "beomi/KcELECTRA-base-v2022"
SEED      = 42
SAMPLE    = 0          # >0 이면 감정별 균형 샘플링(빠른 점검)
os.makedirs(OUT_DIR, exist_ok=True)

EMO6 = ["분노","슬픔","불안","상처","당황","기쁨"]
EMO6_TO_MODE4 = {"분노":"화남","슬픔":"속상","불안":"계획","상처":"응원","당황":"속상","기쁨":"응원"}
EMO6_EN = {"분노":"anger","슬픔":"sadness","불안":"anxiety","상처":"hurt","당황":"fluster","기쁨":"joy"}
MODE4 = ["응원","속상","화남","계획"]
MODE_EN = {"응원":"encourage","속상":"sad","화남":"angry","계획":"plan"}
to_mode4 = lambda L: np.array([EMO6_TO_MODE4[e] for e in L])

# 한글 폰트(있으면) — 혼동행렬 라벨 깨짐 방지
for f in ["Malgun Gothic","AppleGothic","NanumGothic"]:
    try:
        plt.rcParams["font.family"]=f; break
    except Exception: pass
plt.rcParams["axes.unicode_minus"]=False

## 1. 데이터 로드 & 클래스 분포\n불균형 여부를 먼저 본다 — Macro 지표를 메인으로 쓰는 근거.

In [ ]:
rows = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8") if l.strip()]
df = pd.DataFrame(rows)[["text","emotion"]].dropna()
df["emotion"] = df["emotion"].astype(str).str.strip()
df = df[df["emotion"].isin(EMO6)].reset_index(drop=True)
if SAMPLE>0:
    df = df.groupby("emotion", group_keys=False).apply(lambda g: g.sample(min(len(g),SAMPLE), random_state=SEED)).reset_index(drop=True)
print("총", len(df), "건")
vc = df["emotion"].value_counts().reindex(EMO6)
print(vc)
vc.plot(kind="bar", title="6감정 분포"); plt.tight_layout(); plt.show()

## 2. KcELECTRA 임베딩 (freeze)\nmean-pooling + 마지막 4개 레이어 concat + L2 정규화. (GPU 권장)

In [ ]:
def embed_texts(texts, model_name=MODEL_NAME, batch_size=32, cache=None):
    if cache and os.path.exists(cache):
        print("캐시 로드:", cache); return np.load(cache)
    import torch
    from transformers import AutoTokenizer, AutoModel
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print("임베딩:", model_name, "device=", dev)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name, output_hidden_states=True).to(dev).eval()
    vecs, t0 = [], time.time()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = [" ".join(str(x).split()) for x in texts[i:i+batch_size]]
            enc = tok(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(dev)
            out = model(**enc)
            mask = enc["attention_mask"].unsqueeze(-1).float()
            mp = lambda h: (h*mask).sum(1)/mask.sum(1).clamp(min=1)
            emb = torch.cat([mp(h) for h in out.hidden_states[-4:]], dim=-1)
            vecs.append(emb.cpu().numpy())
            if i % (batch_size*20)==0: print(f"  {i}/{len(texts)} ({time.time()-t0:.0f}s)")
    emb = np.vstack(vecs).astype(np.float32)
    emb /= (np.linalg.norm(emb,axis=1,keepdims=True)+1e-9)
    if cache: np.save(cache, emb); print("캐시 저장:", cache, emb.shape)
    return emb

X = embed_texts(df["text"].tolist(), cache=os.path.join(OUT_DIR, f"emb_{SAMPLE or 'all'}.npy"))
y = df["emotion"].values
print("임베딩 shape:", X.shape)

## 3. Train / Test 분할 (층화, seed=42)

In [ ]:
from sklearn.model_selection import train_test_split
Xtr,Xte,ytr,yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print("train", len(Xtr), "/ test", len(Xte))

## 4. 모델 학습 — LogReg / LinearSVM / XGBoost\ndense 임베딩엔 선형·SVM이 트리보다 잘 나오는 경우가 많아 함께 비교한다. 클래스 불균형은 가중치로 보정.

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
import joblib

fitted = {}   # name -> (clf, le or None)

fitted["LogReg"] = (make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced")).fit(Xtr, ytr), None)
fitted["LinearSVM"] = (make_pipeline(StandardScaler(), LinearSVC(class_weight="balanced", C=0.5)).fit(Xtr, ytr), None)

try:
    from xgboost import XGBClassifier
    le = LabelEncoder().fit(EMO6)
    freq = pd.Series(ytr).value_counts()
    w = np.array([len(ytr)/(len(EMO6)*freq[c]) for c in ytr])
    xgb = XGBClassifier(n_estimators=600, max_depth=6, learning_rate=0.08, subsample=0.8,
            colsample_bytree=0.8, tree_method="hist", objective="multi:softprob",
            num_class=len(EMO6), eval_metric="mlogloss", n_jobs=-1, random_state=SEED)
    xgb.fit(Xtr, le.transform(ytr), sample_weight=w)
    fitted["XGBoost"] = (xgb, le)
except ImportError:
    print("xgboost 미설치 — 건너뜀")

print("학습 완료:", list(fitted))

## 5. 평가지표 6종 함수\n① 정확도 ② 혼동행렬 ③ 정밀도 ④ 재현율 ⑤ F1 ⑥ ROC-AUC + 클래스별 리포트

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score)
from sklearn.preprocessing import label_binarize

def predict_and_scores(name, X):
    clf, le = fitted[name]
    if le is not None:                      # XGBoost
        proba = clf.predict_proba(X); pred = le.inverse_transform(proba.argmax(1)); return pred, proba, le.classes_.tolist()
    pred = clf.predict(X)
    if hasattr(clf, "predict_proba"):
        return pred, clf.predict_proba(X), clf.classes_.tolist()
    return pred, clf.decision_function(X), clf.classes_.tolist()   # LinearSVM

def roc_macro(y_true, scores, score_classes, classes):
    if scores is None: return None
    try:
        order = [score_classes.index(c) for c in classes]   # 점수 열을 classes 순서로 정렬
        Yb = label_binarize(y_true, classes=classes)
        return roc_auc_score(Yb, np.asarray(scores)[:, order], average="macro", multi_class="ovr")
    except Exception:
        return None

def full_eval(title, y_true, y_pred, scores, score_classes, classes):
    acc = accuracy_score(y_true, y_pred)
    pre = precision_score(y_true, y_pred, average="macro", labels=classes, zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", labels=classes, zero_division=0)
    f1m = f1_score(y_true, y_pred, average="macro", labels=classes, zero_division=0)
    auc = roc_macro(y_true, scores, score_classes, classes)
    print(f"\n===== {title} =====")
    print(f"① 정확도 Accuracy        : {acc:.4f}")
    print(f"③ 정밀도 Precision(macro) : {pre:.4f}")
    print(f"④ 재현율 Recall(macro)    : {rec:.4f}")
    print(f"⑤ F1-score(macro)        : {f1m:.4f}   <-- 메인 KPI")
    print(f"⑥ ROC-AUC(macro, OvR)    : {auc:.4f}" if auc is not None else "⑥ ROC-AUC               : N/A")
    print("\n[클래스별 정밀도/재현율/F1]")
    print(classification_report(y_true, y_pred, labels=classes, zero_division=0, digits=3))
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    fig, ax = plt.subplots(figsize=(0.9*len(classes)+2, 0.9*len(classes)+1))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=45, ha="right")
    ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9)
    ax.set_title(f"② 혼동행렬 — {title}"); ax.set_xlabel("예측"); ax.set_ylabel("실제")
    plt.tight_layout(); plt.show()
    return {"accuracy":round(acc,4),"precision_macro":round(pre,4),"recall_macro":round(rec,4),
            "f1_macro":round(f1m,4),"roc_auc_macro":(round(auc,4) if auc is not None else None)}

## 6. 6감정 평가

In [ ]:
scores6 = {}
for name in fitted:
    pred, sc, sc_classes = predict_and_scores(name, Xte)
    scores6[name] = full_eval(f"{name} · 6감정", yte, pred, sc, sc_classes, EMO6)

## 7. 4공감모드 평가 (실KPI)\n6감정 예측을 4모드로 매핑해 평가. **4모드 Macro-F1이 메인 KPI**.

In [ ]:
metrics4 = {}
for name in fitted:
    pred, _, _ = predict_and_scores(name, Xte)
    # 4모드는 매핑 후 라벨이라 ROC-AUC 점수 정렬이 모호 → 정확도/P/R/F1/혼동행렬 중심
    metrics4[name] = full_eval(f"{name} · 4모드(KPI)", to_mode4(yte), to_mode4(pred), None, None, MODE4)

## 8. Best 선택 & 저장\n4모드 Macro-F1 기준 best 저장 + metrics.json (백엔드 자동 로드용)

In [ ]:
best = max(metrics4, key=lambda k: metrics4[k]["f1_macro"])
print("BEST =", best, "| 4모드 Macro-F1 =", metrics4[best]["f1_macro"])

import joblib
clf, le = fitted[best]
best_path = ("xgb_emo6.joblib" if le is not None else f"{best.lower()}_emo6.joblib")
joblib.dump(clf, os.path.join(OUT_DIR, best_path))
if le is not None: joblib.dump(le, os.path.join(OUT_DIR, "label_encoder.joblib"))

meta = {"embedding_model":MODEL_NAME,
        "embedding":{"pooling":"mean","last4_concat":True,"l2_norm":True,"max_length":128},
        "emo6":EMO6,"mode4":MODE4,"emo6_to_mode4":EMO6_TO_MODE4,"mode_en":MODE_EN,"emo6_en":EMO6_EN,
        "n_total":int(len(df)),"n_test":int(len(yte)),
        "metrics_6emotion":scores6,"metrics_4mode":metrics4,
        "best_model":best,"best_model_path":best_path,"uses_label_encoder":le is not None}
json.dump(meta, open(os.path.join(OUT_DIR,"metrics.json"),"w",encoding="utf-8"), ensure_ascii=False, indent=2)
print("저장:", sorted(os.listdir(OUT_DIR)))

## 9. 정직한 현황
- **파인튜닝 천장 ≈ 0.74** → freeze-임베딩+분류기는 이 부근(또는 그 이하)을 현실적 목표로 본다. 단일 숫자보다 **클래스별 F1·혼동행렬**로 어느 감정이 약한지 함께 보고.
- **메인 KPI = 4공감모드 Macro-F1** (불균형이라 Accuracy 대신 Macro). 6감정은 참고치.
- 산출물 `metrics.json`은 백엔드(`emotion_model.py`)가 자동 로드 → 챗봇이 LLM 대신 이 모델로 추론. 결과는 메시지의 `emotion6`(TTS)·`emotion_label`(4모드)로 저장.